## デモ

In [17]:
import sys
sys.path.insert(0, '..')
import json
from datetime import datetime
import importlib

from llm_preference_extraction.reasoners import load_implicit_inference_prompt
from llm_preference_extraction.extractors import format_dialogue_for_extraction



### 入力

In [8]:
input_filepath = "../data/output_samples/user00/phase1_logs.jsonl"

In [9]:
with open(input_filepath, "r", encoding="utf-8") as f:
    lines = f.readlines()
    dialogue_id = len(lines)  # 行番号 (1-indexed)
    last_dialogue = json.loads(lines[-1])

print(f"user_id: {last_dialogue['user_id']}")
print(f"conversation_model: {last_dialogue.get('generation_model', '')}")

user_id: user00
conversation_model: gpt-4o


In [10]:
dialogue_text = format_dialogue_for_extraction(last_dialogue["dialogue_history"])

print("\n".join(dialogue_text.split("\n")[:3]))

User: ゲームとジャグリングと研究に時間を使っている. お金はあまり使っておらず, 強いて言えば友人との外食に使っている.
System: それは多様な趣味をお持ちですね。ゲームやジャグリング、研究、それぞれどんなところが好きですか？
User: ゲームは原神で一人で探索したり, league of legends で友達と一緒に遊んだりとゲームを変えれば無限の楽しみ方があるところ. ジャグリングはできる技を増やす事自体楽しいし, サークルメンバーとの交流も魅力の一つ. さらに発表会に向けて練習して当日良いパフォーマンスができると最高に気持ちがいい. 研究は自分の本当に興味があることについて既存の研究を調べ, 知識を深めたうえで更に自分でどう高められるかを検証するのが楽しい. もちろん大変だと感じる瞬間はあるが, やりがいを感じる.


### 設定

In [15]:
from llm_preference_extraction.extractors import create_client, create_few_shot_examples, load_prompt_template,load_schema
from llm_preference_extraction.reasoners import load_implicit_inference_prompt

# 対話生成に使ったモデル
conversation_model = last_dialogue.get('generation_model', '')

# ユーザID
user_id = last_dialogue['user_id']

timestamp = last_dialogue['timestamp']

# few_shot例用データセットを読み込む
dataset_path = "../data/ground_truth/test.json"

with open(dataset_path, "r",encoding="utf-8") as f:
    dataset = json.load(f)

few_shot_id = [0, 1, 2]
model = "llama3.1:8b"
base_url = "http://localhost:11434/v1"
api_key = "llama3.1"

# クライアント作成
client = create_client(base_url, api_key)

# 明示的嗜好抽出用
# プロンプトテンプレート読み込み
few_shot_example_text = create_few_shot_examples(dataset, few_shot_ids=few_shot_id)
prompt_template = load_prompt_template()
explicit_system_prompt = prompt_template.replace("{few_shot_example}", few_shot_example_text)

# スキーマ読み込み
schema = load_schema()

# 暗黙的嗜好推論用
implicit_system_prompt = load_implicit_inference_prompt()

### 嗜好抽出 & 嗜好推論

In [19]:
from llm_preference_extraction.extractors import extract_preferences
from llm_preference_extraction.reasoners import infer_implicit_preferences, integrate_preferences

# 明示的嗜好抽出
explicit_result = extract_preferences(
    client, model, dialogue_text, dialogue_id, explicit_system_prompt, schema
)

# 暗黙的嗜好推論
implicit_result = infer_implicit_preferences(
    client,
    model,
    dialogue_text,
    dialogue_id,
    explicit_result.get("preferences", []),
    implicit_system_prompt
)

integrated_result = integrate_preferences(
    dialogue_id, user_id, explicit_result, implicit_result,
    conversation_model, model
)

integrated_result["source_timestamp"] = timestamp
integrated_result["extraction_timestamp"] = datetime.now().isoformat()

print(f"dialogue_id: {integrated_result['dialogue_id']}")
print(f"user_id: {integrated_result['user_id']}\n")
print(f"conversation_id: {integrated_result['generation_model']}")
print(f"analysis_model: {integrated_result['analysis_model']}")
print(f"explicit_preferences(1例): \n{integrated_result['explicit_preferences'][0]}")
print(f"implicit_preferences(1例): \n{integrated_result['implicit_preferences'][0]}")

dialogue_id: 1
user_id: user00

conversation_id: gpt-4o
analysis_model: llama3.1:8b
explicit_preferences(1例): 
{'combined_axis': 'liking__general', 'entity': 'ゲーム', 'original_mention': 'ゲームは原神で一人で探索したり, league of legends で友達と一緒に遊んだりとゲームを変えれば無限の楽しみ方があるところ.', 'context_tags': ['activity-working_studying'], 'polarity': 'positive', 'intensity': 'high'}
implicit_preferences(1例): 
{'inference': '学業や責任を重視する傾向がある', 'original_mention': '卒論を書き上げなきゃいけないから'}


### 知識グラフ構築

In [26]:
from llm_preference_extraction.graph_builder import build_knowledge_graph, save_knowledge_graph
from pathlib import Path

# integrated_result をリストに包んで渡す
kg_data = build_knowledge_graph(
    preferences_data=[integrated_result],
    user_id=user_id,
    generation_model=conversation_model,
    analysis_model=model,
)

# 結果確認
print(f"トリプレット数: {kg_data['metadata']['total_triples']}")
print(f"  explicit: {kg_data['metadata']['explicit_count']}")
print(f"  implicit: {kg_data['metadata']['implicit_count']}")

# explicit と implicit を1つずつ表示
for type_name in ["explicit", "implicit"]:
    t = next((t for t in kg_data["triples"] if t["type"] == type_name), None)
    if t:
        print(f"[{type_name}]の例")
        print(f"  ({t['head']}) --[{t['relation']}]--> {t['tail']}")
        print(f"  attributes:")
        for k, v in t["attributes"].items():
            print(f"    {k}: {v}")
        print()

トリプレット数: 14
  explicit: 6
  implicit: 8
[explicit]の例
  (user00) --[liking__general]--> ゲーム
  attributes:
    polarity: positive
    intensity: high
    context_tags: ['activity-working_studying']
    original_mention: ゲームは原神で一人で探索したり, league of legends で友達と一緒に遊んだりとゲームを変えれば無限の楽しみ方があるところ.
    dialogue_id: 1
    source_timestamp: 2026-01-24T18:27:05.893313

[implicit]の例
  (user00) --[implicit_preference]--> 学業や責任を重視する傾向がある
  attributes:
    original_mention: 卒論を書き上げなきゃいけないから
    dialogue_id: 1
    source_timestamp: 2026-01-24T18:27:05.893313

